# Chapter 5: Encoding and Evolution
*From: Designing Data-Intensive Applications*

---

## TL;DR

- **Compatibility is a rolling-upgrade problem.** Old code and new code coexist; old data and new data coexist. You need **backward compatibility** (new code reads old data) AND **forward compatibility** (old code reads new data) *at the same time*.
- **Textual formats (JSON, XML, CSV) are ubiquitous but sloppy** about types, binary blobs, and evolution; language-specific formats (Python pickle, Java `Serializable`) are insecure and non-portable.
- **Schema-driven binary formats win on size and evolvability.** Protocol Buffers uses stable **field tag numbers** as the contract; Avro matches fields **by name** using a writer's schema + reader's schema, which is ideal for dynamically generated schemas (e.g., database dumps).
- **Dataflow happens in three modes:** through **databases** (data outlives code → forward compat matters), through **services** (REST, RPC), and **asynchronously** via message brokers, actor systems, or workflow engines.
- **Durable execution** (Temporal-style) upgrades async dataflow with replay-based exactly-once semantics — at the cost of strict determinism requirements and workflow-code versioning.

---

## 1. Why Evolution Matters

Applications change. A new field appears in a record; an old column is renamed; a service starts sending a new response shape. In a large system you cannot freeze the world and upgrade everything atomically — you do a **rolling upgrade**, node by node. So during the rollout:

- Some nodes run **v1**, some run **v2**.
- Some records on disk were written by **v1**, some by **v2**.
- Clients (especially mobile clients and third-party integrations) may be running **v0** from last year.

Two directions of compatibility are required, and both must hold *simultaneously*:

| Direction | Meaning | Usually... |
|---|---|---|
| **Backward compatibility** | New code reads old data. | Easier — new code knows what old data looked like. |
| **Forward compatibility**  | Old code reads new data. | Harder — old code must *gracefully ignore* things it does not understand (unknown fields, new enum values). |

The uncomfortable truth for databases: **data outlives code**. A record written five years ago by a long-deleted version of the app still needs to decode today. That is why storage encodings deserve as much attention as API encodings.

---

## 2. Encoding Size Comparison

The chapter uses one running example — a `Person` record `{userName: "Martin", favoriteNumber: 1337, interests: ["daydreaming", "hacking"]}` — and encodes it four ways. This is the closest thing to "capacity estimation" in a data-encoding chapter: a concrete feel for how much the wire format costs you.

In [ ]:
# Encoding sizes for the running Person record from the chapter:
#   {userName: "Martin", favoriteNumber: 1337, interests: ["daydreaming", "hacking"]}

encodings = [
    ("JSON (textual)",        81),  # whitespace-free, field names as strings
    ("MessagePack (bin JSON)", 66),  # binary-tagged but still carries field names
    ("Protocol Buffers",       33),  # schema-driven: tag numbers, no field names on wire
    ("Avro",                   32),  # schema-driven: no tags, values in schema order
]

json_bytes = encodings[0][1]
avro_bytes = encodings[-1][1]

print(f"{'Encoding':<22} {'Bytes':>6} {'vs Avro':>10} {'vs JSON':>10}")
print("-" * 52)
for name, size in encodings:
    overhead = size - avro_bytes
    reduction_vs_json = (1 - size / json_bytes) * 100
    print(f"{name:<22} {size:>6} {overhead:>+9}B  {reduction_vs_json:>8.1f}%")

print()
print(f"JSON -> Avro compression ratio: {json_bytes / avro_bytes:.2f}x smaller")
print(f"Binary MessagePack saves ~{(1 - encodings[1][1] / json_bytes) * 100:.0f}% vs JSON without a schema.")
print(f"Schema-driven formats cut another ~{(1 - avro_bytes / encodings[1][1]) * 100:.0f}% by dropping field names from the wire.")

**Read this table as follows.** MessagePack gives you a modest win (~18%) by binary-tagging JSON values — but it still ships field names on the wire. The real step-function savings (~60% vs JSON) come when a **schema** is agreed out-of-band and the wire only carries tag numbers (Protobuf) or raw values in schema order (Avro). The cost of those savings is tooling: code generation, schema registries, and explicit compatibility rules.

---

## 3. Textual Formats

Textual formats are everywhere because they are human-readable and trivially inspectable. But they carry well-known baggage.

| Property | Language-specific (pickle / Java Serializable) | JSON | XML | CSV |
|---|---|---|---|---|
| Cross-language portability | No — tied to a runtime | Yes | Yes | Yes |
| Human-readable | No | Yes | Yes | Yes |
| Number type fidelity | Yes | **Ambiguous** (int vs float; >2^53 breaks JS) | Ambiguous (strings) | None (all strings) |
| Binary data | Yes (native) | Must Base64-encode | Must Base64-encode | Awkward |
| Schema support | Implicit (class defs) | Optional (JSON Schema) | XML Schema / DTD | None standard |
| Security | **Unsafe** — decode executes code | Safe | Safe (if entity expansion disabled) | Safe |
| Evolution safety | Brittle | Depends on reader ("ignore unknown fields") | Schema-dependent | Column order/naming fragile |

**JSON Schema** adds validation power — regex, `if/then/else`, `$ref` — at the cost of complexity. The critical evolution lever is `additionalProperties`:

- **`additionalProperties: true` (open)** — unknown fields pass through. Forward-compatible: old validators tolerate new fields.
- **`additionalProperties: false` (closed)** — unknown fields are rejected. Strict but breaks forward compatibility the moment anyone adds a field upstream.

---

## 4. Binary Schema-Driven: Protocol Buffers

Protocol Buffers encodes a record as a concatenation of `(tag, wire_type, value)` triples. The **tag number**, not the field name, is the contract between writer and reader.

```mermaid
graph LR
    subgraph ".proto schema (shared out of band)"
        S1["field 1: userName (string)"]
        S2["field 2: favoriteNumber (int64)"]
        S3["field 3: interests (repeated string)"]
    end
    subgraph "Wire bytes (~33 B)"
        W1["tag=1 type=LEN len=6 'Martin'"]
        W2["tag=2 type=VARINT value=1337"]
        W3["tag=3 type=LEN 'daydreaming'"]
        W4["tag=3 type=LEN 'hacking'"]
    end
    S1 -.tag 1.-> W1
    S2 -.tag 2.-> W2
    S3 -.tag 3.-> W3
    S3 -.tag 3.-> W4
```

**Evolution rules that fall out of this design:**

| Operation | Backward (new reads old) | Forward (old reads new) | How |
|---|---|---|---|
| Add optional field with a new tag | OK | OK | Old readers skip unknown tag; new readers use default/missing value. |
| Remove an optional field | OK | OK | Reserve the tag number so it is never reused. |
| Rename a field | OK | OK | Names are not on the wire — only tags are. |
| Change a field's type | Often breaks | Often breaks | E.g., `int32` → `int64` is safe one way; `string` ↔ `bytes` depends. |
| Make an optional field required | **BREAKS** | **BREAKS** | Old records may be missing it; old writers may omit it. |
| Reuse a retired tag number | **BREAKS** | **BREAKS** | Stale data with the old meaning will be misparsed. |

---

## 5. Binary Schema-Driven: Avro

Avro takes a different bet. There are no tag numbers on the wire — fields are serialized in schema order with their raw bytes. For this to work, the decoder needs **the exact schema that was used to write the bytes**. Avro names this the **writer's schema**, and the decoder provides its own **reader's schema**; the library resolves differences by **field name**.

```mermaid
graph LR
    subgraph "Encode side"
        WApp[Writer app] --> WS[writer's schema v1]
        WApp --> Bytes[(compact bytes: ~32B)]
        WS --> Bytes
    end
    subgraph "Decode side"
        Bytes --> Resolver{{Avro schema resolution<br/>match by field name}}
        RS[reader's schema v2] --> Resolver
        WS -.transmitted with data.-> Resolver
        Resolver --> Decoded[decoded record]
    end

    subgraph "How writer's schema reaches reader"
        F1[Large file: header once]
        F2[DB: per-record version tag + schema registry]
        F3[Network: negotiated at handshake]
    end
```

**Why this matters.** When the schema is *dynamically generated* — say, Avro derived from a relational DB's current column list — assigning and maintaining stable Protobuf tag numbers is impractical. Avro matches by name, so a generated schema just works. The price: you must always pair bytes with their writer's schema, which is exactly what **Avro object container files**, **schema registries**, and **handshake negotiation** solve.

---

## 6. Modes of Dataflow

Encoded data reaches a reader through one of three channels, each with its own compatibility story.

```mermaid
graph TB
    subgraph "1 Database dataflow"
        Writer1[App v2 writer] -->|write record| DB[(Database)]
        DB -->|read record| Reader1[App v1 reader]
        DB -.->|"archived dump; rewrite to latest schema"| Archive[("Analytics warehouse<br/>Avro or Parquet")]
    end

    subgraph "2 Service dataflow (REST/RPC)"
        Client[Client] -->|"request (backward compat)"| Service[Service]
        Service -->|"response (forward compat)"| Client
    end

    subgraph "3 Async dataflow"
        Prod[Producer] --> Broker[(Message broker / topic)]
        Broker --> C1[Consumer A]
        Broker --> C2[Consumer B]
        Actor1[Actor] -.message.-> Actor2[Actor]
        Temporal[Workflow orchestrator] -->|replay log| Temporal
        Temporal --> Worker[Activity worker]
    end
```

The key observation: **a database is a message to your future self**. A record written by yesterday's code must be readable by tomorrow's code AND by last month's straggler client. That is why forward compatibility is not just a client-server concern.

---

## 7. RPC vs Local Calls

An RPC is designed to *look* like a local function call, which is exactly what makes it dangerous. The fallacies of distributed computing live here.

| Aspect | Local function call | RPC |
|---|---|---|
| Predictability | Succeeds or throws deterministically | Can succeed, fail, **or hang** (timeouts required) |
| Failure modes | Exception from callee | Network partition, timeout, duplicate delivery, partial success |
| Latency variance | Nanoseconds, tight distribution | Milliseconds to seconds; **long tail** dominates p99 |
| Argument passing | By reference, zero-copy | By value — must be serialized; cycles and large objects are expensive |
| Cross-language types | Same runtime, shared types | Must agree on a schema (Protobuf, Avro, OpenAPI) or accept loss |
| Idempotence | Irrelevant | **Required** for safe retries |

REST + JSON + OpenAPI dominates public APIs (browser-friendly, cache-friendly, introspectable). gRPC + Protobuf dominates internal service-to-service traffic (compact, codegen'd stubs, strict compatibility rules, streaming). API **versioning** happens through URL paths (`/v2/...`), custom headers, or content negotiation — the point is always to keep old clients working while new ones get new fields.

---

## 8. Service Discovery Options

A client must find an instance to call. The options trade off dynamism, observability, and operational surface area.

| Option | How it works | Dynamism | TLS termination | Observability | Typical use |
|---|---|---|---|---|---|
| Hardware load balancer | Dedicated appliance | Low | Yes | Vendor-specific | Legacy datacenters |
| Software LB (NGINX, HAProxy, Envoy) | Reverse proxy process | Medium (config reload) | Yes | Access logs, metrics | North-south traffic |
| DNS | A/SRV records | Low (TTL-bound) | No (name only) | Minimal | Coarse-grained discovery |
| Registry (etcd, Consul, ZooKeeper) | Services register; clients query | High (seconds) | Client-side | Per-client | Dynamic internal fleets |
| Service mesh (Istio, Linkerd) | Sidecar per pod + control plane | Very high | **Automatic mTLS** | Full east-west tracing | Microservices at scale |

A service mesh is essentially "software LB + registry + policy + observability" folded into every pod. You pay for it in complexity and sidecar overhead.

---

## 9. Durable Execution

When you need a **long-running workflow** that survives crashes and orchestrates many RPCs with exactly-once semantics, a queue is not enough. **Durable execution engines** (Temporal, Cadence, AWS Step Functions) solve this by recording every effectful step to a log and **replaying** that log deterministically when a worker restarts.

```mermaid
sequenceDiagram
    participant O as Workflow orchestrator
    participant L as Event history log
    participant W as Activity worker

    O->>L: append StartWorkflow(input)
    O->>W: invoke activity_charge_card()
    W-->>O: return auth_id
    O->>L: append ActivityCompleted(charge, auth_id)
    Note over O,L: Worker crashes here
    Note over O: New worker picks up and replays log
    O->>L: read history, rehydrate state
    O->>W: invoke activity_ship_order() (next step)
    W-->>O: return tracking_no
    O->>L: append ActivityCompleted(ship, tracking_no)
    O->>L: append WorkflowCompleted
```

The workflow code reads like ordinary imperative code, but every `await activity(...)` is actually "check the log; if this step already completed, return its recorded result; otherwise dispatch to a worker and record the outcome."

In [ ]:
# Pseudo-code illustrating Temporal-style durable execution.
# This is NOT real Temporal — it's an educational skeleton so you can run it.

def order_workflow(order_id: str) -> dict:
    # Each call here would be a durable activity in real Temporal.
    # The engine replays completed steps from the event history on restart.
    auth_id     = f"auth-{order_id}"           # stand-in for charge_card(order_id)
    tracking_no = f"trk-{order_id}"            # stand-in for ship_order(order_id)
    _confirmed  = True                         # stand-in for send_confirmation_email(order_id)
    return {"order_id": order_id, "auth": auth_id, "tracking": tracking_no}

result = order_workflow("A-1001")
print("Workflow result:", result)
print()
print("In a real engine:")
print("  - Each step is logged BEFORE execution and its outcome logged AFTER.")
print("  - On worker crash, a new worker replays the log to reach the current step.")
print("  - Activities MUST be idempotent; workflow code MUST be deterministic.")
print("  - Changing workflow code mid-flight requires versioning (getVersion gates).")

**The sharp edges:** workflow code must be **deterministic** on replay (no `datetime.now()`, no random IDs, no direct I/O — route those through activities), and any code change that reorders steps requires a versioning gate so in-flight workflows replay against the *old* logic while new workflows use the new logic.

---

## 10. Key Takeaways

1. **Two compatibilities, always.** Any non-trivial system needs both backward and forward compatibility at once, because rolling upgrades guarantee that old/new code and old/new data coexist.
2. **Schema on the wire has a cost.** JSON ships field names in every record; binary schema-driven formats move that cost to an out-of-band schema and cut wire size by ~60%.
3. **Protobuf vs Avro is "static vs dynamic."** Protobuf's tag numbers are perfect when humans hand-edit schemas; Avro's name matching is perfect when schemas are generated from something else (a DB, a config).
4. **Data outlives code.** Storage forces forward compatibility on you whether you want it or not — old records must keep decoding. Migrations are expensive; archival re-encoding (Avro object containers, Parquet) is how you pay down that debt.
5. **RPC is not a local call.** Budget for timeouts, partial failures, retries, and idempotence from day one. The network is part of your type system.
6. **Pick the right dataflow mode.** Synchronous RPC for request/response; message brokers for loose coupling and fan-out; durable execution for multi-step workflows that must survive crashes with exactly-once semantics.
7. **Durable execution is powerful and brittle.** Replay-based exactly-once is a real capability, but it constrains how you write (and change) workflow code — determinism and explicit versioning are non-negotiable.

---

## See Also

- [[01-backward-forward-compatibility-and-rolling-upgrades]] — the compatibility contract that every encoding in this chapter must satisfy.
- [[02-textual-encodings-and-json-schema]] — JSON, XML, CSV, and the evolution knobs of JSON Schema.
- [[03-protocol-buffers-field-tags-and-schema-evolution]] — tag+type+value wire format and the evolution rules it enables.
- [[04-avro-writer-and-reader-schemas]] — name-matched resolution for dynamically generated schemas.
- [[05-dataflow-through-databases]] — why forward compatibility is a *storage* concern and when to re-encode archives.
- [[06-rest-rpc-and-service-discovery]] — REST vs gRPC, versioning, and the discovery options from DNS to service mesh.
- [[07-async-dataflow-brokers-actors-durable-execution]] — brokers, actors, and the exactly-once replay model of Temporal-style engines.